In [ ]:
%%sql -r dataframe_1
USE DATABASE EYPROJECT;
USE SCHEMA PUBLIC;

In [ ]:
req_text = """
pystac-client
planetary-computer
odc-stac
rasterio
xarray
rioxarray
geopandas
shapely
pyproj
"""

with open("/tmp/requirements_elevation.txt", "w") as f:
    f.write(req_text.strip())

print("Saved /tmp/requirements_elevation.txt")

In [ ]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
!pip install uv
!uv pip install -r /tmp/requirements_elevation.txt

In [ ]:
import pystac_client
import planetary_computer as pc
import rasterio
from odc.stac import stac_load
import pandas as pd
import numpy as np
from tqdm import tqdm
print("All key packages imported successfully.")

In [ ]:
train_df = pd.read_csv("water_quality_training_dataset.csv")
val_df = pd.read_csv("submission_template.csv")

train_coords = train_df[["Latitude","Longitude"]].drop_duplicates()
val_coords = val_df[["Latitude","Longitude"]].drop_duplicates()

all_coords = pd.concat([train_coords,val_coords]).drop_duplicates().reset_index(drop=True)

print("Unique coordinates:", len(all_coords))

In [ ]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)

collection = "esa-worldcover"

In [ ]:
def get_landcover(lat, lon):

    search = catalog.search(
        collections=[collection],
        intersects={
            "type": "Point",
            "coordinates": [lon, lat]
        }
    )

    items = list(search.items())

    if len(items) == 0:
        return np.nan

    item = items[0]

    asset = item.assets["map"]

    href = pc.sign(asset.href)

    with rasterio.open(href) as src:
        value = list(src.sample([(lon,lat)]))[0][0]

    return int(value)

In [ ]:
landcover = []

for i,row in tqdm(all_coords.iterrows(), total=len(all_coords)):

    lat = row["Latitude"]
    lon = row["Longitude"]

    lc = get_landcover(lat,lon)

    landcover.append(lc)

all_coords["landcover"] = landcover

display(all_coords.head())

In [ ]:
train_lc = train_df.merge(
    all_coords,
    on=["Latitude","Longitude"],
    how="left"
)[["Latitude","Longitude","Sample Date","landcover"]]

val_lc = val_df.merge(
    all_coords,
    on=["Latitude","Longitude"],
    how="left"
)[["Latitude","Longitude","Sample Date","landcover"]]

In [ ]:
train_lc.to_csv("/tmp/landcover_training.csv",index=False)
val_lc.to_csv("/tmp/landcover_validation.csv",index=False)

print("Saved land cover features")

In [ ]:
train_lc

In [ ]:
session.sql("""
PUT file:///tmp/landcover_training.csv @~ AUTO_COMPRESS=FALSE OVERWRITE=TRUE
""").collect()

session.sql("""
PUT file:///tmp/landcover_validation.csv @~ AUTO_COMPRESS=FALSE OVERWRITE=TRUE
""").collect()